In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
"""
Seq2Seq Translation with Φ* Tracking - KAGGLE VERSION
Optimized for Kaggle Notebooks with modular cell structure
Dataset: https://www.kaggle.com/datasets/dhruvildave/en-fr-translation-dataset
"""

# ===============================
# CELL 1: Install & Import Dependencies
# ===============================
import kagglehub
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from collections import Counter
import random
from typing import List, Tuple
import pandas as pd
import os
import gc

print("✓ All libraries imported successfully")
print(f"✓ PyTorch version: {torch.__version__}")
print(f"✓ Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# ===============================
# CELL 2: Configuration & Hyperparameters
# ===============================

print("\n" + "="*70)
print("CONFIGURATION")
print("="*70)

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Model hyperparameters
HIDDEN_SIZE = 256  # Kaggle has more RAM, so we can use larger model
EMBEDDING_SIZE = 256
NUM_LAYERS = 3
DROPOUT = 0.3

# Training hyperparameters
EPOCHS = 200
BATCH_SIZE = 64
LR = 0.001
CLIP = 1
TEACHER_FORCING_RATIO = 0.5

# Data parameters
MAX_LEN = 40
MIN_FREQ = 3
MAX_SAMPLES = 10000  # Kaggle can handle more data

# Φ* parameters
PHI_SAMPLES_PER_EPOCH = 50
PHI_TAU = 1
PHI_N_PARTITIONS = 10
PHI_COMPUTE_FREQUENCY = 2  # Compute every 2 epochs

# Random seed for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

print(f"\nModel Configuration:")
print(f"  Hidden Size: {HIDDEN_SIZE}")
print(f"  Embedding Size: {EMBEDDING_SIZE}")
print(f"  Num Layers: {NUM_LAYERS}")
print(f"  Dropout: {DROPOUT}")

print(f"\nTraining Configuration:")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Learning Rate: {LR}")
print(f"  Max Samples: {MAX_SAMPLES}")

print(f"\nΦ* Configuration:")
print(f"  τ (time lag): {PHI_TAU}")
print(f"  Partition samples: {PHI_N_PARTITIONS}")
print(f"  Compute frequency: Every {PHI_COMPUTE_FREQUENCY} epochs")

print("="*70)

In [ ]:
# ===============================
# CELL 3: Download Dataset
# ===============================

print("\n" + "="*70)
print("DOWNLOADING DATASET")
print("="*70)

# Download latest version using kagglehub
path = kagglehub.dataset_download("dhruvildave/en-fr-translation-dataset")
print(f"✓ Path to dataset files: {path}")

# Find the CSV file
csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
print(f"✓ Found CSV files: {csv_files}")

if csv_files:
    dataset_file = os.path.join(path, csv_files[0])
    print(f"✓ Using dataset: {dataset_file}")
else:
    raise FileNotFoundError("No CSV file found in dataset directory!")

print("="*70)

In [ ]:
# ===============================
# CELL 4: Load & Clean Raw Data 
# ===============================

print("\n" + "="*70)
print("LOADING & CLEANING DATA")
print("="*70)

# Load CSV
print("Loading CSV file...")
df = pd.read_csv(dataset_file)
print(f"✓ Total rows in CSV: {len(df):,}")
print(f"✓ Columns: {list(df.columns)}")

# Display first few rows
print("\nFirst 5 rows:")
print(df.head())

# Check for missing values
print("\nMissing values:")
print(df.isnull().sum())

# Clean data
print("\nCleaning data...")
# Remove rows with missing values
df_clean = df.dropna()
print(f"✓ After removing NaN: {len(df_clean):,} rows")

# Remove duplicates
df_clean = df_clean.drop_duplicates()
print(f"✓ After removing duplicates: {len(df_clean):,} rows")

# Limit to MAX_SAMPLES if needed
if len(df_clean) > MAX_SAMPLES:
    df_clean = df_clean.sample(n=MAX_SAMPLES, random_state=SEED).reset_index(drop=True)
    print(f"✓ Sampled {MAX_SAMPLES:,} rows for training")

# FIXED: Use actual column names from the CSV
# The dataset uses 'en' and 'fr' instead of 'English words/sentences' and 'French words/sentences'
if 'en' in df_clean.columns and 'fr' in df_clean.columns:
    # This is the actual format
    raw_english = df_clean['en'].tolist()
    raw_french = df_clean['fr'].tolist()
    print("✓ Using columns: 'en' and 'fr'")
elif 'English words/sentences' in df_clean.columns:
    # Alternative format (older version)
    raw_english = df_clean['English words/sentences'].tolist()
    raw_french = df_clean['French words/sentences'].tolist()
    print("✓ Using columns: 'English words/sentences' and 'French words/sentences'")
else:
    # Unknown format - show available columns
    print(f"ERROR: Expected columns not found!")
    print(f"Available columns: {list(df_clean.columns)}")
    raise ValueError("Cannot find English and French columns in dataset")

print(f"\n✓ Final dataset size: {len(raw_english):,} sentence pairs")

# Show sample translations
print("\nSample translations (raw):")
for i in range(5):
    print(f"  EN: {raw_english[i]}")
    print(f"  FR: {raw_french[i]}")
    print()

# Clean up
del df, df_clean
gc.collect()

print("="*70)

In [ ]:
# ===============================
# CELL 5: Process & Tokenize Data
# ===============================

print("\n" + "="*70)
print("PROCESSING & TOKENIZING DATA")
print("="*70)

def preprocess_sentence(sentence):
    """Clean and tokenize a sentence"""
    # Convert to lowercase
    sentence = str(sentence).lower().strip()
    
    # Basic cleaning - remove extra spaces
    sentence = ' '.join(sentence.split())
    
    # Tokenize by splitting on spaces
    tokens = sentence.split()
    
    return tokens

# Process all sentences
print("Tokenizing sentences...")
english_sentences = []
french_sentences = []

for en, fr in zip(raw_english, raw_french):
    en_tokens = preprocess_sentence(en)
    fr_tokens = preprocess_sentence(fr)
    
    # Filter by length
    if 2 <= len(en_tokens) <= MAX_LEN and 2 <= len(fr_tokens) <= MAX_LEN:
        english_sentences.append(en_tokens)
        french_sentences.append(fr_tokens)

print(f"✓ Valid sentence pairs after filtering: {len(english_sentences):,}")

# Calculate statistics
en_lengths = [len(s) for s in english_sentences]
fr_lengths = [len(s) for s in french_sentences]

print(f"\nEnglish sentence statistics:")
print(f"  Min length: {min(en_lengths)}")
print(f"  Max length: {max(en_lengths)}")
print(f"  Mean length: {np.mean(en_lengths):.2f}")
print(f"  Median length: {np.median(en_lengths):.2f}")

print(f"\nFrench sentence statistics:")
print(f"  Min length: {min(fr_lengths)}")
print(f"  Max length: {max(fr_lengths)}")
print(f"  Mean length: {np.mean(fr_lengths):.2f}")
print(f"  Median length: {np.median(fr_lengths):.2f}")

# Split into train/validation (80/20)
split_idx = int(0.8 * len(english_sentences))

train_en = english_sentences[:split_idx]
train_fr = french_sentences[:split_idx]
val_en = english_sentences[split_idx:]
val_fr = french_sentences[split_idx:]

print(f"\n✓ Train samples: {len(train_en):,}")
print(f"✓ Validation samples: {len(val_en):,}")

# Show processed examples
print("\nSample tokenized translations:")
for i in range(3):
    print(f"  EN: {' '.join(train_en[i])}")
    print(f"  FR: {' '.join(train_fr[i])}")
    print()

# Clean up raw data
del raw_english, raw_french, english_sentences, french_sentences
gc.collect()

print("="*70)

In [ ]:
# ===============================
# CELL 6: Build Vocabularies
# ===============================

print("\n" + "="*70)
print("BUILDING VOCABULARIES")
print("="*70)

class Vocabulary:
    """Vocabulary for mapping between words and indices"""
    def __init__(self, freq_threshold=MIN_FREQ):
        self.itos = {0: "<PAD>", 1: "<SOS>", 2: "<EOS>", 3: "<UNK>"}
        self.stoi = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2, "<UNK>": 3}
        self.freq_threshold = freq_threshold

    def __len__(self):
        return len(self.itos)

    def build_vocabulary(self, sentence_list):
        """Build vocabulary from list of tokenized sentences"""
        frequencies = Counter()
        idx = 4

        # Count word frequencies
        for sentence in sentence_list:
            for word in sentence:
                frequencies[word] += 1

        # Add words that meet frequency threshold
        for word, count in frequencies.items():
            if count >= self.freq_threshold:
                self.stoi[word] = idx
                self.itos[idx] = word
                idx += 1
        
        print(f"  Total unique words: {len(frequencies)}")
        print(f"  Words above threshold ({self.freq_threshold}): {len(self.stoi) - 4}")

    def numericalize(self, text):
        """Convert tokens to indices"""
        return [self.stoi.get(token, self.stoi["<UNK>"]) for token in text]

# Build English vocabulary
print("\n Building English vocabulary...")
src_vocab = Vocabulary(freq_threshold=MIN_FREQ)
src_vocab.build_vocabulary(train_en)
print(f"✓ English vocab size: {len(src_vocab)}")

# Build French vocabulary
print("\n Building French vocabulary...")
trg_vocab = Vocabulary(freq_threshold=MIN_FREQ)
trg_vocab.build_vocabulary(train_fr)
print(f"✓ French vocab size: {len(trg_vocab)}")

# Calculate vocabulary coverage
def calculate_coverage(vocab, sentences):
    """Calculate % of tokens that are not UNK"""
    total_tokens = 0
    unk_tokens = 0
    
    for sentence in sentences:
        for token in sentence:
            total_tokens += 1
            if token not in vocab.stoi:
                unk_tokens += 1
    
    coverage = (1 - unk_tokens / total_tokens) * 100
    return coverage, unk_tokens, total_tokens

en_cov, en_unk, en_total = calculate_coverage(src_vocab, train_en + val_en)
fr_cov, fr_unk, fr_total = calculate_coverage(trg_vocab, train_fr + val_fr)

print(f"\n Vocabulary coverage:")
print(f"  English: {en_cov:.2f}% ({en_total - en_unk}/{en_total} tokens)")
print(f"  French: {fr_cov:.2f}% ({fr_total - fr_unk}/{fr_total} tokens)")

# Show some vocabulary examples
print(f"\n Sample vocabulary (first 20 English words):")
sample_words = [src_vocab.itos[i] for i in range(4, min(24, len(src_vocab)))]
print(f"  {', '.join(sample_words)}")

print("="*70)

In [ ]:
# ===============================
# CELL 7: Create PyTorch Datasets & DataLoaders
# ===============================

print("\n" + "="*70)
print("CREATING DATASETS & DATALOADERS")
print("="*70)

class TranslationDataset(Dataset):
    """PyTorch Dataset for translation pairs"""
    def __init__(self, src_sentences, trg_sentences, src_vocab, trg_vocab):
        self.src_sentences = src_sentences
        self.trg_sentences = trg_sentences
        self.src_vocab = src_vocab
        self.trg_vocab = trg_vocab

    def __len__(self):
        return len(self.src_sentences)

    def __getitem__(self, idx):
        src = self.src_sentences[idx]
        trg = self.trg_sentences[idx]

        # Add SOS and EOS tokens and convert to indices
        src_numericalized = [self.src_vocab.stoi["<SOS>"]] + \
                           self.src_vocab.numericalize(src) + \
                           [self.src_vocab.stoi["<EOS>"]]

        trg_numericalized = [self.trg_vocab.stoi["<SOS>"]] + \
                           self.trg_vocab.numericalize(trg) + \
                           [self.trg_vocab.stoi["<EOS>"]]

        return (torch.tensor(src_numericalized, dtype=torch.long),
                torch.tensor(trg_numericalized, dtype=torch.long))

def collate_fn(batch):
    """Collate function to pad sequences to same length in batch"""
    src_batch, trg_batch = [], []

    for src, trg in batch:
        src_batch.append(src)
        trg_batch.append(trg)

    # Pad sequences
    src_batch = pad_sequence(src_batch, padding_value=0, batch_first=True)
    trg_batch = pad_sequence(trg_batch, padding_value=0, batch_first=True)

    return src_batch, trg_batch

# Create datasets
print("Creating datasets...")
train_dataset = TranslationDataset(train_en, train_fr, src_vocab, trg_vocab)
val_dataset = TranslationDataset(val_en, val_fr, src_vocab, trg_vocab)

print(f"✓ Train dataset size: {len(train_dataset):,}")
print(f"✓ Validation dataset size: {len(val_dataset):,}")

# Create dataloaders
print("\nCreating dataloaders...")
train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE,
    shuffle=True, 
    collate_fn=collate_fn,
    num_workers=2,  # Kaggle supports parallel loading
    pin_memory=True if torch.cuda.is_available() else False
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE,
    shuffle=False, 
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True if torch.cuda.is_available() else False
)

print(f"✓ Train batches: {len(train_loader)}")
print(f"✓ Validation batches: {len(val_loader)}")

# Test dataloader
print("\n Testing dataloader...")
for src_batch, trg_batch in train_loader:
    print(f"  Source batch shape: {src_batch.shape}")
    print(f"  Target batch shape: {trg_batch.shape}")
    print(f"  Sample source (first sequence): {src_batch[0][:10].tolist()}...")
    print(f"  Sample target (first sequence): {trg_batch[0][:10].tolist()}...")
    break

print("="*70)

In [ ]:
# ===============================
# CELL 8: Define Seq2Seq Model Architecture
# ===============================

print("\n" + "="*70)
print("DEFINING MODEL ARCHITECTURE")
print("="*70)

class Encoder(nn.Module):
    """LSTM Encoder"""
    def __init__(self, input_size, embedding_size, hidden_size, num_layers=1, dropout=0.5):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.embedding = nn.Embedding(input_size, embedding_size)
        self.lstm = nn.LSTM(
            embedding_size, 
            hidden_size, 
            num_layers,
            dropout=dropout if num_layers > 1 else 0,
            batch_first=True
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x shape: (batch_size, seq_len)
        embedding = self.dropout(self.embedding(x))
        # embedding shape: (batch_size, seq_len, embedding_size)
        
        outputs, (hidden, cell) = self.lstm(embedding)
        # outputs shape: (batch_size, seq_len, hidden_size)
        # hidden shape: (num_layers, batch_size, hidden_size)
        # cell shape: (num_layers, batch_size, hidden_size)
        
        return outputs, hidden, cell

class Decoder(nn.Module):
    """LSTM Decoder"""
    def __init__(self, output_size, embedding_size, hidden_size, num_layers=1, dropout=0.5):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.embedding = nn.Embedding(output_size, embedding_size)
        self.lstm = nn.LSTM(
            embedding_size, 
            hidden_size, 
            num_layers,
            dropout=dropout if num_layers > 1 else 0,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_size, output_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, hidden, cell):
        # x shape: (batch_size) or (batch_size, 1)
        x = x.unsqueeze(1) if x.dim() == 1 else x
        # x shape: (batch_size, 1)
        
        embedding = self.dropout(self.embedding(x))
        # embedding shape: (batch_size, 1, embedding_size)
        
        output, (hidden, cell) = self.lstm(embedding, (hidden, cell))
        # output shape: (batch_size, 1, hidden_size)
        
        prediction = self.fc(output.squeeze(1))
        # prediction shape: (batch_size, output_size)
        
        return prediction, hidden, cell

class Seq2Seq(nn.Module):
    """Sequence-to-Sequence Model"""
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        # src shape: (batch_size, src_len)
        # trg shape: (batch_size, trg_len)
        
        batch_size = src.shape[0]
        trg_len = trg.shape[1]
        trg_vocab_size = self.decoder.fc.out_features

        # Store decoder outputs
        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(src.device)

        # Encode source sequence
        encoder_outputs, hidden, cell = self.encoder(src)

        # First input to decoder is SOS token
        input = trg[:, 0]

        for t in range(1, trg_len):
            # Decode one step
            output, hidden, cell = self.decoder(input, hidden, cell)
            outputs[:, t, :] = output

            # Teacher forcing: use actual next token as input
            # Otherwise: use predicted token
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input = trg[:, t] if teacher_force else top1

        return outputs

print("✓ Encoder class defined")
print("✓ Decoder class defined")
print("✓ Seq2Seq class defined")

print("\n Model architecture:")
print(f"  Encoder: {NUM_LAYERS}-layer LSTM")
print(f"    Input: vocab_size={len(src_vocab)}, embed_size={EMBEDDING_SIZE}")
print(f"    Hidden: {HIDDEN_SIZE}")
print(f"  Decoder: {NUM_LAYERS}-layer LSTM")
print(f"    Input: vocab_size={len(trg_vocab)}, embed_size={EMBEDDING_SIZE}")
print(f"    Hidden: {HIDDEN_SIZE}")
print(f"    Output: vocab_size={len(trg_vocab)}")

print("="*70)

In [ ]:
# ===============================
# CELL 9: Initialize Model, Optimizer & Loss
# ===============================

print("\n" + "="*70)
print("INITIALIZING MODEL")
print("="*70)

# Create encoder and decoder
encoder = Encoder(
    input_size=len(src_vocab),
    embedding_size=EMBEDDING_SIZE,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT
)

decoder = Decoder(
    output_size=len(trg_vocab),
    embedding_size=EMBEDDING_SIZE,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT
)

# Create Seq2Seq model
model = Seq2Seq(encoder, decoder).to(device)

# Count parameters
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

total_params = count_parameters(model)
encoder_params = count_parameters(encoder)
decoder_params = count_parameters(decoder)

print(f"\n Model Parameters:")
print(f"  Encoder parameters: {encoder_params:,}")
print(f"  Decoder parameters: {decoder_params:,}")
print(f"  Total parameters: {total_params:,}")

# Initialize optimizer
optimizer = optim.Adam(model.parameters(), lr=LR)
print(f"\n✓ Optimizer: Adam (lr={LR})")

# Initialize loss function
PAD_IDX = trg_vocab.stoi["<PAD>"]
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
print(f"✓ Loss function: CrossEntropyLoss (ignore_index={PAD_IDX})")

# Print model summary
print(f"\n Model Summary:")
print(model)

# Move to device and check
print(f"\n✓ Model moved to: {device}")
if torch.cuda.is_available():
    print(f"  GPU Memory allocated: {torch.cuda.memory_allocated(0) / 1024**2:.2f} MB")
    print(f"  GPU Memory reserved: {torch.cuda.memory_reserved(0) / 1024**2:.2f} MB")

print("="*70)

In [ ]:
# ===============================
# IMPROVED CELL 10: Φ* Estimator (FIXED)
# ===============================

print("\n" + "="*70)
print("INITIALIZING IMPROVED Φ* ESTIMATOR")
print("="*70)

@dataclass
class PhiConfig:
    """Configuration for Φ* estimation"""
    tau: int = PHI_TAU
    n_partitions_sample: int = PHI_N_PARTITIONS

class BarrettSethPhiStar:
    """
    IMPROVED Φ* (Integrated Information) Estimator
    With better numerical stability and debugging
    """
    def __init__(self, config):
        self.config = config

    def gaussian_mi(self, X, Y, debug=False):
        """
        Compute mutual information assuming Gaussian distribution
        MI(X;Y) = 0.5 * log(|Cov_X| * |Cov_Y| / |Cov_XY|)
        
        IMPROVED: Better regularization and numerical stability
        """
        if len(X) < 10:  # Need minimum samples
            return 0.0
            
        n = X.shape[1]
        
        # Stronger regularization for numerical stability
        reg_factor = 1e-4  # Increased from 1e-6
        reg = reg_factor * np.eye(n)

        try:
            # Compute covariance matrices with regularization
            Cov_X = np.cov(X.T) + reg
            Cov_Y = np.cov(Y.T) + reg
            
            # Joint covariance needs more regularization
            joint = np.hstack([X, Y])
            Cov_XY = np.cov(joint.T) + reg_factor * np.eye(2*n)

            # Compute log determinants (more numerically stable than determinants)
            sx, logdet_x = np.linalg.slogdet(Cov_X)
            sy, logdet_y = np.linalg.slogdet(Cov_Y)
            sxy, logdet_xy = np.linalg.slogdet(Cov_XY)

            if debug:
                print(f"    [DEBUG] sign(det): X={sx}, Y={sy}, XY={sxy}")
                print(f"    [DEBUG] log|det|: X={logdet_x:.4f}, Y={logdet_y:.4f}, XY={logdet_xy:.4f}")

            # Check for valid determinants
            if sx <= 0 or sy <= 0 or sxy <= 0:
                if debug:
                    print(f"    [DEBUG] Invalid determinant, returning 0")
                return 0.0

            # Compute mutual information in bits
            MI = 0.5 * (logdet_x + logdet_y - logdet_xy) / np.log(2)
            
            if debug:
                print(f"    [DEBUG] MI = {MI:.6f} bits")
            
            return max(MI, 0.0)  # Ensure non-negative
            
        except np.linalg.LinAlgError as e:
            if debug:
                print(f"    [DEBUG] LinAlgError: {e}")
            return 0.0

    def estimate(self, states, debug=False):
        """
        Estimate Φ* from state trajectory
        
        Φ* = I(X_t; X_{t+τ}) - min_partition [I(A_t; A_{t+τ}) + I(B_t; B_{t+τ})]
        
        IMPROVED: Better error handling and debugging
        """
        if debug:
            print(f"  [DEBUG] Input states shape: {states.shape}")
            print(f"  [DEBUG] Tau: {self.config.tau}")
        
        # Check minimum samples
        if len(states) <= self.config.tau:
            if debug:
                print(f"  [DEBUG] Not enough samples: {len(states)} <= {self.config.tau}")
            return 0.0
        
        # Check variance
        state_std = np.std(states, axis=0)
        if np.mean(state_std) < 1e-6:
            if debug:
                print(f"  [DEBUG] Hidden states have very low variance: {np.mean(state_std):.8f}")
            return 0.0

        # Create time-lagged pairs
        X = states[:-self.config.tau]  # X_t
        Y = states[self.config.tau:]   # X_{t+τ}

        if debug:
            print(f"  [DEBUG] X shape: {X.shape}, Y shape: {Y.shape}")

        # Compute whole-system mutual information
        I_whole = self.gaussian_mi(X, Y, debug=debug)
        
        if debug:
            print(f"  [DEBUG] Whole-system MI: {I_whole:.6f}")
        
        if I_whole <= 1e-8:  # Essentially zero
            return 0.0

        # Find minimum information partition (MIP)
        n_dim = X.shape[1]
        nodes = np.arange(n_dim)
        min_I = np.inf

        for i in range(self.config.n_partitions_sample):
            # Random bipartition
            k = np.random.randint(1, max(2, n_dim//2 + 1))  # Ensure k >= 1
            A = np.random.choice(nodes, k, replace=False)
            B = np.setdiff1d(nodes, A)

            if len(B) == 0:
                continue

            # Compute MI for each partition
            I_A = self.gaussian_mi(X[:, A], Y[:, A], debug=False)
            I_B = self.gaussian_mi(X[:, B], Y[:, B], debug=False)
            
            partition_I = I_A + I_B
            min_I = min(min_I, partition_I)
            
            if debug and i == 0:
                print(f"  [DEBUG] Sample partition: |A|={len(A)}, |B|={len(B)}")
                print(f"  [DEBUG]   I_A={I_A:.6f}, I_B={I_B:.6f}, sum={partition_I:.6f}")

        if min_I == np.inf:
            min_I = 0.0

        # Φ* = integrated information
        phi = max(I_whole - min_I, 0.0)
        
        if debug:
            print(f"  [DEBUG] Min partition MI: {min_I:.6f}")
            print(f"  [DEBUG] Final Φ*: {phi:.6f}")
        
        return phi

def collect_encoder_hidden_states(encoder, dataloader, n_batches=5, device='cuda'):
    """
    Collect hidden states from encoder for Φ* computation
    
    IMPROVED: Collect more samples and add normalization
    """
    states = []
    encoder.eval()

    with torch.no_grad():
        for i, (src, _) in enumerate(dataloader):
            if i >= n_batches:
                break
            
            src = src.to(device)
            outputs, hidden, cell = encoder(src)
            
            # Extract last layer hidden states
            # hidden shape: (num_layers, batch_size, hidden_size)
            for b in range(src.size(0)):
                state = hidden[-1, b, :].cpu().numpy()
                states.append(state)
    
    states = np.array(states)
    
    # Optional: Normalize states (can help with numerical stability)
    # Uncomment if needed:
    # states = (states - states.mean(axis=0)) / (states.std(axis=0) + 1e-8)
    
    return states

# Initialize Φ* estimator
phi_config = PhiConfig(tau=PHI_TAU, n_partitions_sample=PHI_N_PARTITIONS)
phi_estimator = BarrettSethPhiStar(phi_config)

print(" Φ* estimator initialized")
print(f"\n🧠 Φ* Configuration:")
print(f"  Time lag (τ): {phi_config.tau}")
print(f"  Partition samples: {phi_config.n_partitions_sample}")
print(f"  Compute frequency: Every {PHI_COMPUTE_FREQUENCY} epochs")


print("="*70)

In [ ]:
# ===============================
# CELL 11: Training & Evaluation Functions
# ===============================

print("\n" + "="*70)
print("DEFINING TRAINING FUNCTIONS")
print("="*70)

def train_epoch(model, iterator, optimizer, criterion, clip=1):
    """
    Train the model for one epoch
    
    Args:
        model: Seq2Seq model
        iterator: DataLoader for training data
        optimizer: Optimizer
        criterion: Loss function
        clip: Gradient clipping value
    
    Returns:
        float: Average loss for the epoch
    """
    model.train()
    epoch_loss = 0

    for i, (src, trg) in enumerate(iterator):
        src = src.to(device)
        trg = trg.to(device)

        # Zero gradients
        optimizer.zero_grad()

        # Forward pass
        output = model(src, trg, teacher_forcing_ratio=TEACHER_FORCING_RATIO)

        # Reshape for loss calculation
        # output shape: (batch_size, trg_len, vocab_size)
        # We ignore the first token (SOS)
        output_dim = output.shape[-1]
        output = output[:, 1:].reshape(-1, output_dim)
        trg = trg[:, 1:].reshape(-1)

        # Compute loss
        loss = criterion(output, trg)

        # Backward pass
        loss.backward()

        # Clip gradients to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)

        # Update weights
        optimizer.step()

        epoch_loss += loss.item()

    return epoch_loss / len(iterator)

def evaluate(model, iterator, criterion):
    """
    Evaluate the model
    
    Args:
        model: Seq2Seq model
        iterator: DataLoader for evaluation data
        criterion: Loss function
    
    Returns:
        float: Average loss
    """
    model.eval()
    epoch_loss = 0

    with torch.no_grad():
        for src, trg in iterator:
            src = src.to(device)
            trg = trg.to(device)

            # Forward pass without teacher forcing
            output = model(src, trg, teacher_forcing_ratio=0)

            # Reshape for loss calculation
            output_dim = output.shape[-1]
            output = output[:, 1:].reshape(-1, output_dim)
            trg = trg[:, 1:].reshape(-1)

            # Compute loss
            loss = criterion(output, trg)
            epoch_loss += loss.item()

    return epoch_loss / len(iterator)

def compute_phi_star(encoder, dataloader, phi_estimator, n_batches=5):
    """
    Compute Φ* for current encoder state
    
    Args:
        encoder: Encoder network
        dataloader: DataLoader to sample from
        phi_estimator: BarrettSethPhiStar instance
        n_batches: Number of batches to use
    
    Returns:
        float: Φ* value
    """
    # Collect hidden states
    states = collect_encoder_hidden_states(
        encoder, 
        dataloader, 
        n_batches=n_batches,
        device=device
    )
    
    # Estimate Φ*
    phi = phi_estimator.estimate(states)
    
    # Clean up
    del states
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    return phi

print("✓ train_epoch() defined")
print("✓ evaluate() defined")
print("✓ compute_phi_star() defined")

print("\n Training process:")
print("  1. Forward pass through encoder-decoder")
print("  2. Compute cross-entropy loss (ignoring PAD tokens)")
print("  3. Backward pass and gradient clipping")
print("  4. Update weights with Adam optimizer")
print("  5. Compute Φ* every N epochs by:")
print("     - Collecting encoder hidden states")
print("     - Computing integrated information")

print("="*70)

In [ ]:
# ===============================
# IMPROVED CELL 12: Training Loop with Debug
# ===============================

print("\n" + "="*70)
print("STARTING TRAINING WITH Φ* TRACKING (IMPROVED)")
print("="*70)

# Storage for metrics
train_loss_history = []
val_loss_history = []
phi_history = []
epochs_list = []

# Training header
print(f"\n{'Epoch':<8} {'Train Loss':<12} {'Val Loss':<12} {'Φ*':<14} {'Time':<10} {'Note':<10}")
print("="*90)

import time

last_phi = 0.0  # Store last computed Φ*

for epoch in range(1, EPOCHS + 1):
    start_time = time.time()
    
    # Train for one epoch
    train_loss = train_epoch(model, train_loader, optimizer, criterion, clip=CLIP)
    
    # Evaluate on validation set
    val_loss = evaluate(model, val_loader, criterion)
    
    # Store losses
    train_loss_history.append(train_loss)
    val_loss_history.append(val_loss)
    epochs_list.append(epoch)
    
    # Compute Φ* periodically
    should_compute_phi = (epoch % PHI_COMPUTE_FREQUENCY == 0) or (epoch == 1) or (epoch == EPOCHS)
    
    if should_compute_phi:
        # Collect hidden states with MORE batches for better estimate
        n_batches_for_phi = max(5, PHI_SAMPLES_PER_EPOCH // BATCH_SIZE)
        
        states = collect_encoder_hidden_states(
            encoder, 
            train_loader, 
            n_batches=n_batches_for_phi,
            device=device
        )
        
        #Debug on first epoch
        if epoch == 1:
            print(f"\n  [DEBUG INFO FOR EPOCH 1]")
            print(f"  Collected {states.shape[0]} hidden states")
            print(f"  Hidden dim: {states.shape[1]}")
            print(f"  State std: {np.std(states, axis=0).mean():.6f}")
            phi = phi_estimator.estimate(states, debug=True)
            print(f"  Computed Φ* = {phi:.6f}\n")
        else:
            phi = phi_estimator.estimate(states, debug=False)
        
        last_phi = phi
        note = "◆ Φ*"
        
        # Clean up
        del states
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    else:
        phi = last_phi  # Reuse last computed value
        note = ""
    
    phi_history.append(phi)
    
    # Calculate epoch time
    epoch_time = time.time() - start_time
    
    # Print progress
    print(f"{epoch:<8} {train_loss:<12.4f} {val_loss:<12.4f} {phi:<14.6f} {epoch_time:<10.2f}s {note:<10}")
    
    # Memory cleanup every 5 epochs
    if epoch % 5 == 0:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print("="*90)
print("✓ Training complete!")

# Check if Φ* is all zeros
if all(p == 0 for p in phi_history):
    print("\n" + "="*70)
    print("⚠️  WARNING: All Φ* values are 0!")
    print("="*70)
    print("Possible causes:")
    print("  1. Hidden states not varying enough")
    print("  2. Too few samples collected")
    print("  3. Numerical instability")
    print("  4. Model architecture issue")
    print("\nSuggested fixes:")
    print("  • Increase n_batches_for_phi (currently set dynamically)")
    print("  • Run the DIAGNOSTIC CELL to identify the issue")
    print("  • Try increasing HIDDEN_SIZE")
    print("  • Check if encoder is actually training (check train_loss)")
    print("="*70 + "\n")

# Print summary statistics
print("\n" + "="*70)
print("TRAINING SUMMARY")
print("="*70)

best_train_epoch = np.argmin(train_loss_history) + 1
best_val_epoch = np.argmin(val_loss_history) + 1

# Only compute phi stats if not all zeros
if not all(p == 0 for p in phi_history):
    best_phi_epoch = np.argmax(phi_history) + 1
    print(f"\n Best Performance:")
    print(f"  Best train loss: {min(train_loss_history):.4f} (epoch {best_train_epoch})")
    print(f"  Best val loss: {min(val_loss_history):.4f} (epoch {best_val_epoch})")
    print(f"  Max Φ*: {max(phi_history):.6f} (epoch {best_phi_epoch})")
else:
    print(f"\n Best Performance:")
    print(f"  Best train loss: {min(train_loss_history):.4f} (epoch {best_train_epoch})")
    print(f"  Best val loss: {min(val_loss_history):.4f} (epoch {best_val_epoch})")
    print(f"  Max Φ*: N/A (all values are 0)")

print(f"\n Final Metrics:")
print(f"  Final train loss: {train_loss_history[-1]:.4f}")
print(f"  Final val loss: {val_loss_history[-1]:.4f}")
print(f"  Final Φ*: {phi_history[-1]:.6f}")

if not all(p == 0 for p in phi_history):
    print(f"\n Correlation Analysis:")
    correlation = np.corrcoef(phi_history, val_loss_history)[0, 1]
    print(f"  Correlation(Φ*, val_loss): {correlation:.3f}")

    if correlation < -0.3:
        print(f"  → Strong negative correlation: Higher Φ* → Lower loss")
    elif correlation > 0.3:
        print(f"  → Strong positive correlation: Higher Φ* → Higher loss")
    else:
        print(f"  → Weak correlation")

# Improvement statistics
train_improvement = ((train_loss_history[0] - train_loss_history[-1]) / train_loss_history[0]) * 100
val_improvement = ((val_loss_history[0] - val_loss_history[-1]) / val_loss_history[0]) * 100

print(f"\n Improvement from Start:")
print(f"  Train loss: {train_improvement:.1f}% reduction")
print(f"  Val loss: {val_improvement:.1f}% reduction")

if not all(p == 0 for p in phi_history) and phi_history[0] > 0:
    phi_change = ((phi_history[-1] - phi_history[0]) / phi_history[0]) * 100
    print(f"  Φ*: {phi_change:+.1f}% change")

print("="*70)

In [ ]:
# ===============================
# CELL 13: Visualization of Results
# ===============================

print("\n" + "="*70)
print("GENERATING VISUALIZATIONS")
print("="*70)

# Create comprehensive visualization
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Seq2Seq Training with Φ* (Integrated Information) Tracking', 
             fontsize=16, fontweight='bold', y=1.00)

epochs_array = np.array(epochs_list)

# Plot 1: Φ* vs Epoch
ax1 = axes[0, 0]
ax1.plot(epochs_array, phi_history, marker='o', color='#2E86AB', 
         linewidth=2.5, markersize=6, markeredgecolor='white', markeredgewidth=1)
ax1.set_xlabel("Epoch", fontsize=11, fontweight='bold')
ax1.set_ylabel("Φ* (Integrated Information)", fontsize=11, fontweight='bold')
ax1.set_title("Integrated Information vs Training Progress", fontsize=12, fontweight='bold')
ax1.grid(alpha=0.3, linestyle='--')
ax1.set_xlim(0, EPOCHS + 1)

# Highlight max Φ*
max_phi_idx = np.argmax(phi_history)
ax1.scatter(epochs_array[max_phi_idx], phi_history[max_phi_idx], 
           color='red', s=150, marker='*', zorder=5, 
           label=f'Max Φ*={phi_history[max_phi_idx]:.4f}')
ax1.legend(fontsize=9)

# Plot 2: Loss vs Epoch
ax2 = axes[0, 1]
ax2.plot(epochs_array, train_loss_history, marker='s', label='Train Loss',
         linewidth=2.5, markersize=5, color='#F77F00', markeredgecolor='white', markeredgewidth=1)
ax2.plot(epochs_array, val_loss_history, marker='^', label='Val Loss',
         linewidth=2.5, markersize=5, color='#06A77D', markeredgecolor='white', markeredgewidth=1)
ax2.set_xlabel("Epoch", fontsize=11, fontweight='bold')
ax2.set_ylabel("Cross-Entropy Loss", fontsize=11, fontweight='bold')
ax2.set_title("Translation Loss vs Training Progress", fontsize=12, fontweight='bold')
ax2.legend(fontsize=10, loc='upper right')
ax2.grid(alpha=0.3, linestyle='--')
ax2.set_xlim(0, EPOCHS + 1)

# Highlight best validation
best_val_idx = np.argmin(val_loss_history)
ax2.scatter(epochs_array[best_val_idx], val_loss_history[best_val_idx],
           color='red', s=150, marker='*', zorder=5)

# Plot 3: Φ* vs Validation Loss (Scatter)
ax3 = axes[0, 2]
scatter = ax3.scatter(val_loss_history, phi_history, 
                     alpha=0.7, s=100, c=epochs_array, cmap='viridis',
                     edgecolors='black', linewidth=0.5)
ax3.set_xlabel("Validation Loss", fontsize=11, fontweight='bold')
ax3.set_ylabel("Φ*", fontsize=11, fontweight='bold')
ax3.set_title("Φ* vs Validation Performance", fontsize=12, fontweight='bold')
ax3.grid(alpha=0.3, linestyle='--')
cbar = plt.colorbar(scatter, ax=ax3)
cbar.set_label('Epoch', fontsize=10)

# Add trend line
z = np.polyfit(val_loss_history, phi_history, 1)
p = np.poly1d(z)
x_trend = np.linspace(min(val_loss_history), max(val_loss_history), 100)
ax3.plot(x_trend, p(x_trend), "r--", alpha=0.5, linewidth=2, label='Trend')
ax3.legend(fontsize=9)

# Plot 4: Φ* vs Train Loss (Scatter)
ax4 = axes[1, 0]
scatter2 = ax4.scatter(train_loss_history, phi_history, 
                      alpha=0.7, s=100, c=epochs_array, cmap='plasma',
                      edgecolors='black', linewidth=0.5)
ax4.set_xlabel("Train Loss", fontsize=11, fontweight='bold')
ax4.set_ylabel("Φ*", fontsize=11, fontweight='bold')
ax4.set_title("Φ* vs Train Performance", fontsize=12, fontweight='bold')
ax4.grid(alpha=0.3, linestyle='--')
cbar2 = plt.colorbar(scatter2, ax=ax4)
cbar2.set_label('Epoch', fontsize=10)

# Add trend line
z2 = np.polyfit(train_loss_history, phi_history, 1)
p2 = np.poly1d(z2)
x_trend2 = np.linspace(min(train_loss_history), max(train_loss_history), 100)
ax4.plot(x_trend2, p2(x_trend2), "r--", alpha=0.5, linewidth=2, label='Trend')
ax4.legend(fontsize=9)

# Plot 5: Loss Improvement
ax5 = axes[1, 1]
train_improvement_curve = [(train_loss_history[0] - tl) / train_loss_history[0] * 100 
                           for tl in train_loss_history]
val_improvement_curve = [(val_loss_history[0] - vl) / val_loss_history[0] * 100 
                         for vl in val_loss_history]

ax5.plot(epochs_array, train_improvement_curve, marker='o', 
         label='Train Improvement', linewidth=2.5, markersize=5, color='#F77F00')
ax5.plot(epochs_array, val_improvement_curve, marker='o', 
         label='Val Improvement', linewidth=2.5, markersize=5, color='#06A77D')
ax5.set_xlabel("Epoch", fontsize=11, fontweight='bold')
ax5.set_ylabel("Improvement (%)", fontsize=11, fontweight='bold')
ax5.set_title("Cumulative Loss Improvement", fontsize=12, fontweight='bold')
ax5.legend(fontsize=10)
ax5.grid(alpha=0.3, linestyle='--')
ax5.axhline(y=0, color='black', linestyle='-', linewidth=0.8, alpha=0.3)
ax5.set_xlim(0, EPOCHS + 1)

# Plot 6: Φ* Change Rate
ax6 = axes[1, 2]
phi_change_rate = [0] + [phi_history[i] - phi_history[i-1] for i in range(1, len(phi_history))]
ax6.bar(epochs_array, phi_change_rate, color='#2E86AB', alpha=0.7, edgecolor='black', linewidth=0.5)
ax6.set_xlabel("Epoch", fontsize=11, fontweight='bold')
ax6.set_ylabel("Δ Φ*", fontsize=11, fontweight='bold')
ax6.set_title("Φ* Change per Epoch", fontsize=12, fontweight='bold')
ax6.axhline(y=0, color='black', linestyle='-', linewidth=0.8)
ax6.grid(alpha=0.3, linestyle='--', axis='y')
ax6.set_xlim(0, EPOCHS + 1)

plt.tight_layout()
plt.savefig('seq2seq_phi_analysis.png', dpi=300, bbox_inches='tight')
print("✓ Comprehensive plot saved as: seq2seq_phi_analysis.png")
plt.show()

# Additional plot: Loss landscape
fig2, ax = plt.subplots(1, 1, figsize=(10, 6))
ax.plot(epochs_array, train_loss_history, marker='o', label='Train Loss',
        linewidth=3, markersize=6, color='#F77F00', alpha=0.8)
ax.plot(epochs_array, val_loss_history, marker='s', label='Val Loss',
        linewidth=3, markersize=6, color='#06A77D', alpha=0.8)

# Twin axis for Φ*
ax2 = ax.twinx()
ax2.plot(epochs_array, phi_history, marker='^', label='Φ*',
         linewidth=3, markersize=6, color='#2E86AB', alpha=0.8)

ax.set_xlabel('Epoch', fontsize=13, fontweight='bold')
ax.set_ylabel('Loss', fontsize=13, fontweight='bold', color='black')
ax2.set_ylabel('Φ* (Integrated Information)', fontsize=13, fontweight='bold', color='#2E86AB')

ax.set_title('Training Dynamics: Loss and Integrated Information', 
             fontsize=14, fontweight='bold')
ax.grid(alpha=0.3, linestyle='--')
ax.legend(loc='upper right', fontsize=11)
ax2.legend(loc='upper center', fontsize=11)

ax.set_xlim(0, EPOCHS + 1)
ax2.tick_params(axis='y', labelcolor='#2E86AB')

plt.tight_layout()
plt.savefig('seq2seq_dual_axis.png', dpi=300, bbox_inches='tight')
print("✓ Dual-axis plot saved as: seq2seq_dual_axis.png")
plt.show()

print("="*70)

In [ ]:
# ===============================
# Save Training Results to CSV
# ===============================

import os
import pandas as pd

print("\n" + "="*70)
print("SAVING TRAINING RESULTS TO CSV")
print("="*70)

# -------------------------------
# Create save directory
# -------------------------------
SAVE_DIR = "/kaggle/working/phi_results"  


os.makedirs(SAVE_DIR, exist_ok=True)
print(f" Save directory created: {SAVE_DIR}")

# -------------------------------
# -------------------------------
epochs_array = np.array(epochs_list)
phi_array = np.array(phi_history)
train_loss_array = np.array(train_loss_history)
val_loss_array = np.array(val_loss_history)

# -------------------------------
# 1. Φ* vs Epoch
# -------------------------------
df_phi_epoch = pd.DataFrame({
    "epoch": epochs_array,
    "phi_star": phi_array
})
csv_phi_epoch = f"{SAVE_DIR}/phi_vs_epoch.csv"
df_phi_epoch.to_csv(csv_phi_epoch, index=False)
print(f" Saved: {csv_phi_epoch}")

# -------------------------------
# 2. Loss vs Epoch
# -------------------------------
df_loss_epoch = pd.DataFrame({
    "epoch": epochs_array,
    "train_loss": train_loss_array,
    "val_loss": val_loss_array
})
csv_loss_epoch = f"{SAVE_DIR}/loss_vs_epoch.csv"
df_loss_epoch.to_csv(csv_loss_epoch, index=False)
print(f" Saved: {csv_loss_epoch}")

# -------------------------------
# 3. Φ* vs Validation Loss
# -------------------------------
df_phi_val = pd.DataFrame({
    "val_loss": val_loss_array,
    "phi_star": phi_array
})
csv_phi_val = f"{SAVE_DIR}/phi_vs_val_loss.csv"
df_phi_val.to_csv(csv_phi_val, index=False)
print(f" Saved: {csv_phi_val}")

# -------------------------------
# 4. Φ* vs Train Loss
# -------------------------------
df_phi_train = pd.DataFrame({
    "train_loss": train_loss_array,
    "phi_star": phi_array
})
csv_phi_train = f"{SAVE_DIR}/phi_vs_train_loss.csv"
df_phi_train.to_csv(csv_phi_train, index=False)
print(f" Saved: {csv_phi_train}")

# -------------------------------
# 5. Complete Training Summary (ALL DATA)
# -------------------------------
df_summary = pd.DataFrame({
    "epoch": epochs_array,
    "train_loss": train_loss_array,
    "val_loss": val_loss_array,
    "phi_star": phi_array
})
csv_summary = f"{SAVE_DIR}/training_summary.csv"
df_summary.to_csv(csv_summary, index=False)
print(f"✓ Saved: {csv_summary}")

# -------------------------------
# 6. Statistics Summary
# -------------------------------
stats = {
    "total_epochs": [len(epochs_array)],
    "initial_train_loss": [train_loss_array[0]],
    "final_train_loss": [train_loss_array[-1]],
    "best_train_loss": [train_loss_array.min()],
    "initial_val_loss": [val_loss_array[0]],
    "final_val_loss": [val_loss_array[-1]],
    "best_val_loss": [val_loss_array.min()],
    "initial_phi_star": [phi_array[0]],
    "final_phi_star": [phi_array[-1]],
    "max_phi_star": [phi_array.max()],
    "mean_phi_star": [phi_array.mean()],
    "std_phi_star": [phi_array.std()],
    "correlation_phi_val_loss": [np.corrcoef(phi_array, val_loss_array)[0, 1]],
    "correlation_phi_train_loss": [np.corrcoef(phi_array, train_loss_array)[0, 1]]
}

df_stats = pd.DataFrame(stats)
csv_stats = f"{SAVE_DIR}/training_statistics.csv"
df_stats.to_csv(csv_stats, index=False)
print(f"✓ Saved: {csv_stats}")

# -------------------------------
# 7. Display summary table
# -------------------------------
print("\n" + "="*70)
print("SAVED FILES SUMMARY")
print("="*70)

files_created = [
    "phi_vs_epoch.csv",
    "loss_vs_epoch.csv", 
    "phi_vs_val_loss.csv",
    "phi_vs_train_loss.csv",
    "training_summary.csv",
    "training_statistics.csv"
]

print("\n Files created in:", SAVE_DIR)
for i, filename in enumerate(files_created, 1):
    print(f"  {i}. {filename}")

# Display first few rows of summary
print("\n Training Summary Preview (first 10 rows):")
print(df_summary.head(10).to_string(index=False))

print("\n Training Summary Preview (last 10 rows):")
print(df_summary.tail(10).to_string(index=False))

# Display statistics
print("\n Training Statistics:")
print(df_stats.T.to_string())

print("\n" + "="*70)
print(" ALL RESULTS SAVED SUCCESSFULLY!")
print("="*70)

In [ ]:
# ===============================
# CELL 15: Test Translation
# ===============================

print("\n" + "="*70)
print("TESTING TRANSLATION")
print("="*70)

def translate_sentence(model, sentence, src_vocab, trg_vocab, device='cuda', max_len=50):
    """
    Translate a single English sentence to French
    
    Args:
        model: Trained Seq2Seq model
        sentence: Input sentence (string)
        src_vocab: Source vocabulary
        trg_vocab: Target vocabulary
        device: Device to use
        max_len: Maximum generation length
    
    Returns:
        list: Translated tokens
    """
    model.eval()

    # Preprocess and tokenize
    tokens = sentence.lower().strip().split()
    
    # Convert to indices
    tokens = [src_vocab.stoi["<SOS>"]] + \
             src_vocab.numericalize(tokens) + \
             [src_vocab.stoi["<EOS>"]]
    
    # Convert to tensor
    src_tensor = torch.LongTensor(tokens).unsqueeze(0).to(device)

    with torch.no_grad():
        # Encode
        encoder_outputs, hidden, cell = model.encoder(src_tensor)

    # Start decoding
    trg_indexes = [trg_vocab.stoi["<SOS>"]]

    for i in range(max_len):
        trg_tensor = torch.LongTensor([trg_indexes[-1]]).to(device)

        with torch.no_grad():
            output, hidden, cell = model.decoder(trg_tensor, hidden, cell)

        pred_token = output.argmax(1).item()
        trg_indexes.append(pred_token)

        # Stop if EOS token is generated
        if pred_token == trg_vocab.stoi["<EOS>"]:
            break

    # Convert indices to tokens
    trg_tokens = [trg_vocab.itos[i] for i in trg_indexes]

    # Remove SOS and EOS
    return trg_tokens[1:-1] if trg_tokens[-1] == '<eos>' else trg_tokens[1:]

# Test sentences
print("\n🔤 Sample Translations:\n")

test_sentences = [
    "hello",
    "good morning",
    "how are you",
    "thank you very much",
    "i love you",
    "what is your name",
    "goodbye",
    "see you tomorrow",
    "have a good day",
    "where is the bathroom"
]

for sent in test_sentences:
    translation = translate_sentence(model, sent, src_vocab, trg_vocab, device=device)
    print(f"{'EN:':<5} {sent}")
    print(f"{'FR:':<5} {' '.join(translation)}")
    print()

# Test with actual validation examples
print("="*70)
print("VALIDATION SET EXAMPLES")
print("="*70)

print("\n🎯 Actual translations from validation set:\n")

num_examples = 5
for i in range(num_examples):
    en_tokens = val_en[i]
    fr_tokens = val_fr[i]
    
    # Get model prediction
    predicted = translate_sentence(
        model, 
        ' '.join(en_tokens), 
        src_vocab, 
        trg_vocab, 
        device=device
    )
    
    print(f"Example {i+1}:")
    print(f"  Source (EN):      {' '.join(en_tokens)}")
    print(f"  Target (FR):      {' '.join(fr_tokens)}")
    print(f"  Predicted (FR):   {' '.join(predicted)}")
    
    # Simple word-level accuracy
    matches = sum(1 for a, b in zip(fr_tokens, predicted) if a == b)
    accuracy = matches / max(len(fr_tokens), len(predicted)) * 100
    print(f"  Word accuracy:    {accuracy:.1f}%")
    print()

print("="*70)

# Interactive translation (optional)
print("\n💬 Try your own translations:")
print("   (Enter sentences below, or skip to continue)")
print("="*70)

try:
    while True:
        user_input = input("\nEnter English sentence (or 'quit' to stop): ").strip()
        
        if user_input.lower() in ['quit', 'exit', 'q', '']:
            break
        
        translation = translate_sentence(
            model, 
            user_input, 
            src_vocab, 
            trg_vocab, 
            device=device
        )
        
        print(f"Translation: {' '.join(translation)}")
        
except KeyboardInterrupt:
    print("\n\nInteractive mode stopped.")

print("\n✓ Translation testing complete!")
print("="*70)

In [ ]:
# ===============================
# CELL 16: Detailed Φ* Analysis
# ===============================

print("\n" + "="*70)
print("DETAILED Φ* ANALYSIS")
print("="*70)

print("\n What does Φ* tell us about our model?\n")

# Analyze Φ* trajectory
phi_trend = np.polyfit(epochs_list, phi_history, 1)
phi_slope = phi_trend[0]

print("1. Φ* TRAJECTORY:")
if phi_slope > 0.001:
    print(f"   ↗ Φ* is INCREASING (slope: {phi_slope:.6f})")
    print("   → The encoder is developing MORE integrated representations")
    print("   → Hidden states are becoming more mutually dependent")
elif phi_slope < -0.001:
    print(f"   ↘ Φ* is DECREASING (slope: {phi_slope:.6f})")
    print("   → The encoder is developing MORE modular representations")
    print("   → Hidden states are becoming more independent")
else:
    print(f"   → Φ* is STABLE (slope: {phi_slope:.6f})")
    print("   → The encoder maintains consistent integration level")

# Analyze correlation with performance
corr_phi_val = np.corrcoef(phi_history, val_loss_history)[0, 1]
corr_phi_train = np.corrcoef(phi_history, train_loss_history)[0, 1]

print(f"\n2. CORRELATION WITH PERFORMANCE:")
print(f"   Φ* vs Val Loss: {corr_phi_val:+.3f}")
print(f"   Φ* vs Train Loss: {corr_phi_train:+.3f}")

if abs(corr_phi_val) > 0.5:
    if corr_phi_val < 0:
        print("   → STRONG negative correlation")
        print("   → Higher integration → Better performance")
        print("   → The model benefits from integrated representations")
    else:
        print("   → STRONG positive correlation")
        print("   → Higher integration → Worse performance")
        print("   → The model benefits from modular representations")
elif abs(corr_phi_val) > 0.3:
    print("   → MODERATE correlation")
    print("   → Φ* and performance are somewhat related")
else:
    print("   → WEAK correlation")
    print("   → Φ* and performance are largely independent")

# Analyze Φ* variance
phi_std = np.std(phi_history)
phi_mean = np.mean(phi_history)
phi_cv = phi_std / phi_mean if phi_mean > 0 else 0

print(f"\n3. Φ* STABILITY:")
print(f"   Mean Φ*: {phi_mean:.6f}")
print(f"   Std Dev: {phi_std:.6f}")
print(f"   Coefficient of Variation: {phi_cv:.3f}")

if phi_cv < 0.1:
    print("   → Very STABLE Φ* across training")
elif phi_cv < 0.3:
    print("   → MODERATE stability")
else:
    print("   → HIGH variability in Φ*")

# Identify phases of training
print(f"\n4. TRAINING PHASES:")

# Early phase (first 1/3)
early_cutoff = len(phi_history) // 3
early_phi_mean = np.mean(phi_history[:early_cutoff])
early_loss_mean = np.mean(val_loss_history[:early_cutoff])

# Middle phase
mid_start = early_cutoff
mid_end = 2 * early_cutoff
mid_phi_mean = np.mean(phi_history[mid_start:mid_end])
mid_loss_mean = np.mean(val_loss_history[mid_start:mid_end])

# Late phase
late_phi_mean = np.mean(phi_history[mid_end:])
late_loss_mean = np.mean(val_loss_history[mid_end:])

print(f"   Early (epochs 1-{early_cutoff}):")
print(f"     Avg Φ*: {early_phi_mean:.6f}, Avg Val Loss: {early_loss_mean:.4f}")
print(f"   Middle (epochs {early_cutoff+1}-{mid_end}):")
print(f"     Avg Φ*: {mid_phi_mean:.6f}, Avg Val Loss: {mid_loss_mean:.4f}")
print(f"   Late (epochs {mid_end+1}-{len(phi_history)}):")
print(f"     Avg Φ*: {late_phi_mean:.6f}, Avg Val Loss: {late_loss_mean:.4f}")

# Compare phases
phi_early_to_late = ((late_phi_mean - early_phi_mean) / early_phi_mean) * 100
loss_early_to_late = ((early_loss_mean - late_loss_mean) / early_loss_mean) * 100

print(f"\n   Phase Comparison (Early → Late):")
print(f"     Φ* change: {phi_early_to_late:+.1f}%")
print(f"     Loss improvement: {loss_early_to_late:+.1f}%")

# Peak analysis
print(f"\n5. PEAK PERFORMANCE ANALYSIS:")

# Find epoch with best val loss
best_val_epoch_idx = np.argmin(val_loss_history)
phi_at_best_val = phi_history[best_val_epoch_idx]

# Find epoch with max Φ*
max_phi_epoch_idx = np.argmax(phi_history)
loss_at_max_phi = val_loss_history[max_phi_epoch_idx]

print(f"   At best validation loss (epoch {best_val_epoch_idx + 1}):")
print(f"     Val Loss: {val_loss_history[best_val_epoch_idx]:.4f}")
print(f"     Φ* at that point: {phi_at_best_val:.6f}")

print(f"\n   At maximum Φ* (epoch {max_phi_epoch_idx + 1}):")
print(f"     Φ*: {phi_history[max_phi_epoch_idx]:.6f}")
print(f"     Val Loss at that point: {loss_at_max_phi:.4f}")

if best_val_epoch_idx == max_phi_epoch_idx:
    print("\n   ✓ Best performance and max Φ* occur at SAME epoch!")
    print("   → Strong alignment between integration and performance")
else:
    epoch_diff = abs(best_val_epoch_idx - max_phi_epoch_idx)
    print(f"\n   → {epoch_diff} epoch(s) apart")
    if epoch_diff <= 3:
        print("   → Close alignment between integration and performance")
    else:
        print("   → Integration and performance peaks are separated")

# Information-theoretic interpretation
print(f"\n6. INFORMATION-THEORETIC INTERPRETATION:")
print(f"   Φ* measures how much the encoder's hidden state at time t")
print(f"   predicts its state at time t+{PHI_TAU}, beyond what individual")
print(f"   neuron groups can predict independently.")
print(f"\n   Current Φ* = {phi_history[-1]:.6f} bits")
print(f"   → The system has {phi_history[-1]:.6f} bits of integrated information")
print(f"   → This is information that exists at the whole-system level")
print(f"   → It cannot be reduced to independent subsystems")

# Create a detailed analysis table
analysis_df = pd.DataFrame({
    'Metric': [
        'Φ* Mean',
        'Φ* Std Dev',
        'Φ* Min',
        'Φ* Max',
        'Φ* Final',
        'Φ* Slope',
        'Corr(Φ*, ValLoss)',
        'Best Val Loss',
        'Φ* at Best Val',
        'Max Φ*',
        'ValLoss at Max Φ*'
    ],
    'Value': [
        f'{phi_mean:.6f}',
        f'{phi_std:.6f}',
        f'{min(phi_history):.6f}',
        f'{max(phi_history):.6f}',
        f'{phi_history[-1]:.6f}',
        f'{phi_slope:.6f}',
        f'{corr_phi_val:.3f}',
        f'{min(val_loss_history):.4f}',
        f'{phi_at_best_val:.6f}',
        f'{max(phi_history):.6f}',
        f'{loss_at_max_phi:.4f}'
    ]
})

print(f"\n" + "="*70)
print("SUMMARY TABLE")
print("="*70)
print(analysis_df.to_string(index=False))

# Save detailed analysis
analysis_filename = 'phi_star_analysis.csv'
analysis_df.to_csv(analysis_filename, index=False)
print(f"\n✓ Detailed analysis saved to: {analysis_filename}")

print("="*70)